# V55 - Cheap feature screening

Goal: screen many cheap feature views under one grouped protocol before any full build. New since v50: Indonesian number words in assessments are decoded instead of mean-imputed, and a training-free content view (user chat vs module syllabus cosine) joins the screen. Output is a ranked table with bootstrap CIs, not a submission. Winners go to v56 composition.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics.pairwise import cosine_similarity

SEED = 555
N_SPLITS = 5
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v55")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Load and decode

Assessment answers may be Indonesian number words. Decode first, impute only what is truly missing, and report how much the fix recovers.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
assert len(catalog) == 17
Y = train[MODS].to_numpy(float)
n_train = len(train)

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A_raw = np.array([[num(blobs[u].get(q)) for q in questions] for u in list(train["user_id"]) + test_ids])
print(f"assessment NaN after decode: {np.isnan(A_raw).mean():.4f} (was imputed blindly before v55)")
A = np.where(np.isnan(A_raw), np.nanmean(A_raw, axis=0), A_raw)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
A_tr, A_te = A[:n_train], A[n_train:]

by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
all_users = list(train["user_id"]) + test_ids
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

## Screen each view

Every view gets identical grouped folds and a paired bootstrap CI against the chat baseline. Content matching needs no training: cosine between user chat and each module syllabus.

In [ ]:
def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, Xte=None, alpha=3.0):
    oof = np.zeros_like(Y)
    pred = np.zeros((len(Xte), len(MODS))) if Xte is not None else None
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for tr, va in skf.split(Xtr, strata, fold_groups):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha, random_state=SEED)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
            if pred is not None:
                pred[:, j] += m.predict(Xte) / N_SPLITS
    return oof, pred

results = {}
T12 = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3).fit_transform(docs)
oof_chat, _ = ridge_oof(T12[:n_train].toarray())
results["chat word+bigram"] = ndcg5(Y, oof_chat)
T1 = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs)
oof_w1, _ = ridge_oof(T1[:n_train].toarray())
results["chat word-only"] = ndcg5(Y, oof_w1)
oof_a, _ = ridge_oof(A_tr)
results["assessment decoded"] = ndcg5(Y, oof_a)
results["popularity"] = ndcg5(Y, np.tile(Y.mean(0), (n_train, 1)))

syllabi = [(m, (catalog.set_index("module_id").loc[m, "description_and_syllabus"] if m in set(catalog["module_id"]) else "")) for m in MODS]
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1).fit(docs + [s for _, s in syllabi])
U = vec.transform(docs[:n_train])
M = vec.transform([s for _, s in syllabi])
content = cosine_similarity(U, M)
results["content cosine (train-free)"] = ndcg5(Y, content)
oof_stack, _ = ridge_oof(np.hstack([A_tr, content]))
results["assessment + content"] = ndcg5(Y, oof_stack)

ref = results["chat word+bigram"]
print(f"{'view':28s} {'OOF':>7s} {'delta':>8s} {'CI95':>22s}")
table = []
for name, s in sorted(results.items(), key=lambda kv: -kv[1].mean()):
    d = s - ref
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    print(f"{name:28s} {s.mean():7.5f} {d.mean():+8.5f} [{lo:+.5f},{hi:+.5f}]")
    table.append({"view": name, "oof": round(float(s.mean()), 5), "delta": round(float(d.mean()), 5),
                    "ci_lo": round(float(lo), 5), "ci_hi": round(float(hi), 5)})
pd.DataFrame(table).to_csv(OUT / "v55_screen.csv", index=False)
print("wrote", OUT / "v55_screen.csv")